**Lab5 Ex2 By Prajakta Ghorpade SCFP125004**

Exercise 2 — Research Paper Abstract to Layperson Summary Pipeline
Build a 2-step pipeline. Step 1 extracts the research question, method, and key finding from a paper abstract as structured data. Step 2 generates a plain-language summary for a non-expert audience, using only the structured version from Step 1, not the original abstract.

Input: abstract_text

Output: the structured extraction from Step 1, and the layperson summary from Step 2

Step 1 — Template: PromptTemplate  |  Parser: PydanticOutputParser (fields: research_question, method, key_finding)

Step 2 — Template: ChatPromptTemplate (system persona: "science communicator explaining to a general audience")  |  Parser: StrOutputParser




Step 1— Install the required libraries

In [35]:
%pip install -q langchain-core langchain-openai pydantic
%pip install -q langchain-openai
%pip install -qU langchain-groq


Step 2— Import the required classes

In [36]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_openai import ChatOpenAI

Step 3 — Configure the language model

In [37]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

In [38]:
import os

key = os.environ.get("GROQ_API_KEY")

print("Key exists:", bool(key))
print("Starts with gsk_:", key.startswith("gsk_"))
print("Length:", len(key))
print("Has extra spaces:", key != key.strip())

Key exists: True
Starts with gsk_: False
Length: 3
Has extra spaces: False


In [39]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0
)

response = llm.invoke("Explain research in one simple sentence.")
print(response.content)

Research is a systematic way of asking questions and finding answers to learn new things.


Step 4 — Define the structured output

In [40]:
class ResearchExtraction(BaseModel):
    research_question: str = Field(
        description="The main research question"
    )
    method: str = Field(
        description="The method used in the research"
    )
    key_finding: str = Field(
        description="The main finding reported in the abstract"
    )

parser = PydanticOutputParser(
    pydantic_object=ResearchExtraction
)

Step 5 — Create Step 1: Abstract extraction

In [41]:
extract_prompt = PromptTemplate(
    template="""
Extract the research question, method, and key finding
from the research paper abstract below.

Use only the information given in the abstract.
Do not invent missing information.

Abstract:
{abstract_text}

{format_instructions}
""",
    input_variables=["abstract_text"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

extraction_chain = extract_prompt | llm | parser

Step 6 — Create Step 2: Layperson summary

In [42]:
summary_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a science communicator explaining research "
        "to a general audience. Use simple English. Explain "
        "the research question, method, and key finding clearly. "
        "Use only the supplied structured information. Do not "
        "add facts or conclusions that are not provided."
    ),
    (
        "human",
        "Write a plain-language summary using only this "
        "structured research information:\n{structured_data}"
    )
])

summary_chain = summary_prompt | llm | StrOutputParser()

Step 7 — Run the complete pipeline

In [43]:
abstract_text = """
This study investigated whether a six-week walking program
improved sleep quality among adults with poor sleep.
Researchers assigned 100 adults to either a walking program
or a usual-activity group. Sleep quality was measured before
and after the six weeks. The walking group showed greater
improvement in sleep quality than the usual-activity group.
"""
# Step 1: Extract structured information
extracted = extraction_chain.invoke({
    "abstract_text": abstract_text
})



print("STEP 1: STRUCTURED EXTRACTION")
print(extracted.model_dump_json(indent=2))

# Step 2: Generate a summary using ONLY Step 1's output
structured_data = extracted.model_dump_json()

summary = summary_chain.invoke({
    "structured_data": structured_data
})

print("\nSTEP 2: LAYPERSON SUMMARY")
print(summary)

STEP 1: STRUCTURED EXTRACTION
{
  "research_question": "Does a six-week walking program improve sleep quality among adults with poor sleep?",
  "method": "Randomly assigned 100 adults to either a walking program or a usual-activity group, measuring sleep quality before and after the six-week intervention.",
  "key_finding": "The walking group showed greater improvement in sleep quality than the usual-activity group."
}

STEP 2: LAYPERSON SUMMARY
**Plain‑Language Summary**

- **What the study asked:**  
  The researchers wanted to know if walking for six weeks could make sleep better for adults who normally have poor sleep.

- **How they did it:**  
  They took 100 adults and randomly split them into two groups.  
  * One group followed a walking program for six weeks.  
  * The other group kept doing whatever they normally did (the usual‑activity group).  
  They measured each person’s sleep quality before the program started and again after the six weeks.

- **What they found:**  
  T